# 02 - Data preprocessing | Ames Housing 

**Project**

### Imports and raw dataset
The dataset is loaded from `data/raw/ames.csv`. No transformations are fitted before splitting.

In [ ]:
from pathlib import Path 
import json, joblib 
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

# Support running Jupyter from the repository root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "ames.csv").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "ames.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"


if not RAW_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {RAW_PATH}. Add data/raw/ames.csv first.")
df = pd.read_csv(RAW_PATH)
df.head()

### Define the target variable 

In [ ]:
TARGET = "SalePrice"
#Verificate if the target is in the dataframe and ensensial conditions for the target variable
assert TARGET in df, f"Expected target {TARGET!r}. Found:{df.columns.tolist()}"
assert df[TARGET].notna().all(), "Target contains nulls: investigate before proceeding."
assert (df[TARGET]>0).all(), "log1p assumes nonnegative target; review unexpected values."
assert df.index.is_unique, "Expected unique index; found duplicates."

#Print the target variable information
print("Raw shape:", df.shape)
df.info()

print("----------------------------------------------")
print("----------------------------------------------")
print("Statistical summary:")
print("Target skew:", round(df[TARGET].skew(), 3))
print("Fraction of rows containing any missing predictor:",
      f"{df.drop(columns=TARGET).isna().any(axis=1).mean():.1%}")
print("Most missing predictors (descriptive only):")
print(df.drop(columns=TARGET).isna().mean().sort_values(ascending=False).head(8).to_string())

## Data quality 
From the EDA, we expect mixed numerical/categorical features and missing values. Some missing values describe the absence of a feature (for example, no garage). For this baseline, categorical missing values are treated as an explicit Missing category. We do not drop rows or remove outliers without evidence.

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Missing target column: {TARGET}")
if df[TARGET].isna().any() or (df[TARGET] <= 0).any():
    raise ValueError("SalePrice must be present and strictly positive")

print(f"SalePrice skewness: {df[TARGET].skew():.2f}")
print("Top 10 columns by missing rate:")
display(df.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_rate").style.format({"missing_rate": "{:.1%}"}))

**Missingness policy:** Do not drop features solely because they have many missing values: absence of a pool, fence, or alley can be informative. Numeric features use train-fitted median imputation; categorical features use an explicit missing category. This is a reproducible baseline, not a claim that all missingness is equivalent.

### Select train/test split

In [ ]:
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=SEED)
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

print(f"Train: {len(df_train)} rows, Test: {len(df_test)} rows ")
assert len(df_train) + len(df_test) == len(df), "Train/test split does not match original data"

### Define predictors and target
Remove row identifiers, and treat `MS SubClass` as a categorical feature. Keep target values on the original dollar scale; log-transforming the target, if desired, belongs in the model pipeline.

In [ ]:
#this list contains the columns that will be dropped from the dataframe, if they exist in the dataframe
DROP_COLS = [c for c in ["Order", "PID", "Unnamed: 0"] if c in df.columns]

#Split the data into features and target variable for both train and test sets
X_train = df_train.drop(columns=[TARGET, *DROP_COLS]).copy()
X_test = df_test.drop(columns=[TARGET, *DROP_COLS]).copy()
y_train = df_train[TARGET].copy()
y_test = df_test[TARGET].copy()

#Convertir MS SubClass(categorical category) to string if it exists in the dataframe
if "MS SubClass" in X_train.columns:
    X_train["MS SubClass"] = X_train["MS SubClass"].astype(str)
    X_test["MS SubClass"] = X_test["MS SubClass"].astype(str)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

### Training-only exploratory correlations
These correlations are descriptive and must not be computed on the held-out test set.

In [ ]:
correlations = X_train[numeric_features].corrwith(y_train).dropna()
display(
    correlations.reindex(correlations.abs().sort_values(ascending=False).index)
    .head(10)
    .rename("pearson_r")
    .to_frame()
)

### Fit the preprocessing pipeline on training data only
The same fitted transformer is applied to the test set and must be reused for API inference. Unknown categories are ignored by the one-hot encoder.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape: ", X_test_processed.shape)

In [ ]:
preprocessor

### Validate the output

In [ ]:
from scipy import sparse

def all_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return np.isfinite(values).all()

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert all_finite(X_train_processed) and all_finite(X_test_processed)

print("Checks passed ✓")
print("Final feature count:", len(preprocessor.get_feature_names_out()))

### Persist reproducible artifacts
Save the raw train/test partitions, split indices, fitted transformer, and a JSON schema describing expected input columns. **Never fit the preprocessor again on incoming API requests.**

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(PROCESSED_DIR / "train.csv", index=False)
df_test.to_csv(PROCESSED_DIR / "test.csv", index=False)
np.savez(PROCESSED_DIR / "split_indices.npz", train=train_idx, test=test_idx)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Saved:")
print("- data/processed/train.csv")
print("- data/processed/test.csv")
print("- data/processed/split_indices.npz")
print("- models/preprocessor.joblib")

# API contract: expected raw input columns and categorical casting rules.
schema = {
    "target": TARGET,
    "drop_columns": DROP_COLS,
    "feature_columns": X_train.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "categorical_cast_to_string": ["MS SubClass"] if "MS SubClass" in X_train.columns else [],
    "n_processed_features": int(X_train_processed.shape[1]),
}
with (MODELS_DIR / "input_schema.json").open("w", encoding="utf-8") as handle:
    json.dump(schema, handle, indent=2)
print("- models/input_schema.json")

### Next step: API integration
For an inference endpoint, validate the input against `models/input_schema.json`, convert `MS SubClass` to string, order columns using `feature_columns`, and call the **saved** transformer. Once a trained regression model is available, expose a prediction endpoint. EDA summaries should be computed separately and never refit preprocessing artifacts.